# Snowflake DCM Projects (PrPr) - Quickstart 2 (Workspaces)

⚠️ Note: This project will only work if your account is already enrolled in the Private Preview of Snowflake DCM Projects.

---

## Project Files

This Quickstart contains two demo DCM Projects for you to explore core capabilities:
- **DCM_Platform_Demo** — Platform infrastructure (warehouses, roles, grants, ingestion with a Task loading csv files from a stage into raw tables)
- **DCM_Pipeline_Demo** — Data transformation pipeline (Dynamic Tables, Views, Data Quality Expectations)

Explore the content of the manifest files and the various definition files:
- The Manifest contains different target profiles and templating configurations
- The definition files contain various DEFINE statements for new entities, as well as GRANTS, ATTACH DMF and examples for jinja templating
- The macros folder is optional and can store global jinja macros to be used across different definition files

**Tip:** Use the split-screen to keep manifest and definition files left and this setup file on the right

#

## Role Setup

**Step 1** (optional, but recommended): 

Create a dedicated dcm-developer role to manage DCM Projects on a DEV environment

In [ ]:
use role ACCOUNTADMIN;

create role if not exists DCM_ADMIN;
set USER_NAME = (select current_user());
grant role DCM_ADMIN to user identifier($USER_NAME);

**Step 2:** Grant permissions to create new account infrastructure through DCM deployments
(depending on your needs and project purpose)

In [ ]:
grant CREATE WAREHOUSE on account to role DCM_ADMIN;
grant CREATE ROLE on account to role DCM_ADMIN;
grant CREATE DATABASE on account to role DCM_ADMIN;
grant EXECUTE MANAGED TASK on account to role DCM_ADMIN;
grant EXECUTE TASK on account to role DCM_ADMIN;

grant MANAGE GRANTS on account to role DCM_ADMIN;

**Step 3:** If you want to define and test data quality expectations you also need:

In [ ]:
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role DCM_ADMIN;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role DCM_ADMIN;
grant database role SNOWFLAKE.DATA_METRIC_USER to role DCM_ADMIN;
grant execute data metric function on account to role DCM_ADMIN with grant option;

## Platform Project Setup

**Step 1** (optional): In case you don't have a warehouse available or want to run the deployments on a dedicated warehouse

(DCM commands are mostly meta-data changes, which makes them very lightweight. Only few commands actually require a warehouse and XS is sufficient)

In [ ]:
create warehouse if not exists DCM_WH
with
    warehouse_size = 'XSMALL'
    auto_suspend = 300
    comment = 'For Quickstart Demo of DCM Projects'
;

**Step 2:** Create the DCM Project Object inside a schema that executes and stores all deployments of your project definitions

In [ ]:
%%sql -r create_db_schema
create database if not exists DCM_DEMO;
create schema if not exists DCM_DEMO.PROJECTS;

In [ ]:
use role DCM_ADMIN;

create dcm project if not exists DCM_DEMO.PROJECTS.DCM_PLATFORM_PROJECT_DEV
    comment = 'for DCM Platform Demo - Quickstart 2'
;

## Dry-run a deployment using the PLAN command

**1. In the DCM control panel above the tabs, select the project `DCM_Platform_Demo`**

- See how the `DCM_DEV` target is already pre-selected in top right as it is set as default in the manifest
- Click on the target profile to see that it uses the previously created DCM_PLATFORM_PROJECT_DEV object and the `DEV` templating configuration
- Override the templating value for `user` with your own user-name
- Optional: open the manifest file to see what other values this configuration contains

**2. Execute a "Plan" to the DCM_DEV target environment**
- Click 'Plan' and wait for the definitions to render, compile and dry-run the statements

**3. Review the plan result** 

(if the plan output does not show as a new tab, click again on the blue "Plan" button)

## Deploy the Project Definitions

If the plan result was successful and all planned changes are correct, you can now safely `Deploy` those changes.

- In the top-right corner of your PLAN results tab - click "Deploy".
- Optional: You can add a Deployment-alias which you can think of as a "commit name" which you can see later in the deployment history.
- DCM will create all the objects and attach grants and expectations using the owner-role of the project object.
- Once the deployment completed successfully, you can refresh the Database Explorer on the left to see the created database and all objects inside.

## Load Sample Data

Now that the INGEST schema was created, we can copy some sample CSV files from the workspace to the ingestion stage and trigger the load task.

In [ ]:
%%sql -r stage_result
-- Copy all CSV files from demo repo to the stage
COPY FILES INTO 
    @DCM_DEMO_2.INGEST.DCM_SAMPLE_DATA
FROM 
    'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_2/sample_data'
DETAILED_OUTPUT = TRUE
;

Now that we have some new data in our source stage we can manually trigger the Task to load that new data into our RAW tables.

In [ ]:
%%sql -r load_data
execute task DCM_DEMO_2.INGEST.LOAD_NEW_DATA;

---
# DCM Projects for Pipelines

You can leverage additional DCM capabilities when creating and deploying transformation pipelines that leverage Dynamic Tables and Data Metric Functions.

## Pipeline Project Setup

* The DCM_ADMIN has already created a FINANCE_ADMIN role as part of deploying the DCM Platform Project.
* The FINANCE_ADMIN has also been granted specific privileges that are needed to now operate with it DCM_DEMO_2_FINANCE database.
* Now the FINANCE_ADMIN can create his own DCM Project inside this database to manage a new transformation pipeline
* The FINANCE_ADMIN also has been granted SELECT privileges on the RAW tables. So he can now use these as a data source for the transformation pipeline.

In [ ]:
%%sql -r create_pipeline_project
use role FINANCE_ADMIN;

create or replace dcm project FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV
    comment = 'for DCM Pipeline Demos - Quickstart 2'
;

In [ ]:
-- because DCM can not yet grant APPLICATION ROLES we need to manually grant this here for now  

use role ACCOUNTADMIN;

grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_VIEWER to role FINANCE_ADMIN;
grant application role SNOWFLAKE.DATA_QUALITY_MONITORING_ADMIN to role FINANCE_ADMIN;
grant database role SNOWFLAKE.DATA_METRIC_USER to role FINANCE_ADMIN;
grant execute data metric function on account to role FINANCE_ADMIN;

### PLAN and DEPLOY
Now in the shoes of the FINANCE_ADMIN, you can test the definitions of the Pipeline project with the DEV configuration against the current state on the account, and then deploy.

**1. In the DCM control panel above the tabs, select the project `DCM_Pipeline_Demo`**

**2. Execute a "Plan" to the DCM_DEV target environment**
- Click 'Plan' and wait for the definitions to render, compile and dry-run the statements

**3. Review the plan result** 
- if the plan was successful, you can see the objects that will be created by the FINANCE_ADMIN inside the finance database

**4. Execute the deployment of the Pipeline project** 

### REFRESH Command

Now that the pipeline project was created you can refresh all Dynamic Tables defined in the pipeline project to process the new data from the RAW tables.

Trigger a manual refresh now so you don't have to wait for the scheduled refresh

In [ ]:
%%sql -r refresh_result
execute dcm project FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV 
    REFRESH ALL

    -- optional flow operator to format the json output as table
    ->> select
            f.value:data_timestamp ::string as TIMESTAMP_IN_MS,
            f.value:dt_name ::string as DT_NAME,
            f.value:statistics ::string as RESULT,
            f.value:refreshed_dt_count ::number as REFRESHED_DTS
        from
            $1 as REFRESH_RESULT,
            lateral flatten(input => parse_json(REFRESH_RESULT."result"):refreshed_tables) f
;

## TEST Command

Now that the sample data was loaded and processed throughout the transformation pipeline, you can test all Data Quality Expectations that were set on table-objects inside the project.

Run the command below:

In [ ]:
%%sql -r pipeline_test_results
execute dcm project FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV 
    TEST ALL

    -- optional flow operator to format the json output as table
    ->> select
            f.value:table_name ::string as TABLE_NAME,
            f.value:metric_database ::string as METRIC_DATABASE,
            f.value:metric_schema ::string as METRIC_SCHEMA,
            f.value:metric_name ::string as METRIC_NAME,
            f.value:expectation_name ::string as EXPECTATION_NAME,
            f.value:expectation_expression ::string as EXPECTATION_EXPRESSION,
            case 
                when f.value:expectation_violated ::boolean = 'FALSE' then '✅ MET'
                when f.value:expectation_violated ::boolean = 'TRUE' then '🚫 FAILED'
                end as EXPECTATION_RESULT,
            f.value:value ::number as VALUE
        from
            $1 as TEST_RESULTS,
            lateral flatten(input => parse_json(TEST_RESULTS."message"):expectations) f
;

## PREVIEW Command

If you are making changes to pipeline definitions then you can preview a data-sample even before deploying the change.

Preview can be run for a defined table / dynamic table / view.

For example you can change the definition of a Dynamic Table, then run the PREVIEW command to see how the change impacts the downstream data.

In [ ]:
%%sql -r preview_result
execute dcm project FINANCE_DEV.PROJECTS.DCM_PIPELINE_DEV
    PREVIEW DCM_PIPELINE_DEV.GOLD.FACT_PROSPECT
    using configuration DEV (user => 'YOUR_USER_NAME')
    from
        'snow://workspace/USER$.PUBLIC."snowflake_dcm_projects"/versions/live/Quickstarts/DCM_Project_Quickstart_2/DCM_Pipeline_Demo'
;

---
# Plan and Deploy to a Production Environment

Review the **DCM Projects Quickstart 1** to see how to 
* set up dedicated PROD_DEPLOYER roles 
* create a dedicated DCM Project object for the Prod target
* deploy to the Prod target environment